# Create ``Connectivity Group" based on patch-clamp sampling
This notebook takes a ConnectivityMatrix and a ConnectivityGroup describing the results of multi-patch experiments as inputs.
It then uses the multi-patch experiment as a reference for the number of simultaneously sampled neurons and replicates that experiment on the ConnectivityMatrix. The results are written to a ConnectivityGroup that can then be separately analyzed.

In [ ]:
import numpy
import pandas
import conntility
import tqdm

# INPUTS
fn_conn_mat = "Rat_functional_conmat.h5"
fn_sampling_ref = "peng_et_al_human_multi_patch_grp.h5"

# OUTPUT
fn_out = "virtual_patch_clamp_results.h5"

# PARAMETERS
# which coordinate describes the vertical dimension in the ConnectivityMatrix?
con_mat_col_y = "y"
# names of the two other coordinates
con_mat_cols_slice = ["x", "z"]
# virtual slicing parameters
slice_angle = 0.0
slice_position = 0.0
slice_thickness = 125.0
# patch sample parameters
# 1. By how many um do we vary the center around which we sample?
sample_variability_um = 150.0
# 2. Spatial scale of the sampling. This one is crucial!
sample_scale_um = 220.0


# EXECUTE: LOAD DATA
M = conntility.ConnectivityMatrix.from_h5(fn_conn_mat)
P = conntility.ConnectivityGroup.from_h5(fn_sampling_ref)

### Perform the sampling

In [5]:
# How many neurons were sampled in each experiment?
n_smpl = [len(P[i]) for i in P.index]

G = []
# Build a slice
S = M.slice(slice_angle, slice_position, slice_thickness, columns_slice=con_mat_cols_slice, column_y=con_mat_col_y)
for n in tqdm.tqdm(n_smpl):
    g = S.patch_sample(n, numpy.random.rand(2) * sample_variability_um - (sample_variability_um/2),  # Add random offset as parameterized
                   [[sample_scale_um ** 2, 0], [0, sample_scale_um ** 2]], columns_xy=["slice_x", "slice_y"])
    G.append(g)

100%|██████████| 231/231 [00:01<00:00, 139.98it/s]


### Write the results

In [57]:
grp = conntility.ConnectivityGroup(pandas.DataFrame({"index": range(len(G))}), G)
grp.to_h5("/Users/mwr/Documents/artefacts/human_model_multi_patch_grp_peng-like3.h5")